In [1]:
import numpy as np
import pandas as pd 
import matplotlib.pyplot as plt
import seaborn as sns

print("libraries imported successfully")

libraries imported successfully


In [2]:
import os, json, gzip, re
from pathlib import Path
from urllib.parse import urljoin, urlparse

import pandas as pd
from bs4 import BeautifulSoup
from tqdm.auto import tqdm

%pip install requests pandas beautifulsoup4 tqdm seaborn matplotlib

/Users/margedeleon/Library/Python/3.9/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


# Reddit Data Collection Setup

Before running the code below:
1. Create a Reddit developer account at https://www.reddit.com/prefs/apps
2. Create a new application and get your client_id and client_secret
3. Replace the placeholders in the code with your actual credentials
4. Install the PRAW library using pip

In [ ]:
# Install required package
%pip install praw

import praw
import datetime
import time
import pandas as pd
import numpy as np
from tqdm import tqdm

# Initialize Reddit API client
reddit = praw.Reddit(
    client_id="margaretcdeleon",  # Replace with your Reddit API client ID
    client_secret="YOUR_CLIENT_SECRET",  # Replace with your Reddit API client secret
    user_agent="PSLF_Research_Bot/1.0",  # Custom user agent
)

def scrape_subreddit(subreddit_name, start_date, end_date=None):
    """
    Scrape posts from a subreddit within a specified date range
    """
    subreddit = reddit.subreddit(subreddit_name)
    posts_data = []
    
    if end_date is None:
        end_date = datetime.datetime.now()
    
    for post in tqdm(subreddit.new(limit=None)):
        post_date = datetime.datetime.fromtimestamp(post.created_utc)
        
        if post_date < start_date:
            break
            
        if post_date <= end_date:
            post_data = {
                'id': post.id,
                'title': post.title,
                'text': post.selftext,
                'author': str(post.author),
                'score': post.score,
                'upvote_ratio': post.upvote_ratio,
                'num_comments': post.num_comments,
                'created_utc': post.created_utc,
                'subreddit': subreddit_name,
                'url': post.url,
                'permalink': post.permalink
            }
            posts_data.append(post_data)
            
    return pd.DataFrame(posts_data)

In [ ]:
# Define the subreddits to scrape
subreddits = ['Teachers', 'teaching']

# Set the date range for data collection
start_date = datetime.datetime(2020, 1, 1)  # Start from January 1, 2020
end_date = datetime.datetime.now()  # Current date

# Initialize an empty list to store all dataframes
all_data = []

# Scrape data from each subreddit
for subreddit in subreddits:
    print(f"Scraping r/{subreddit}...")
    df = scrape_subreddit(subreddit, start_date, end_date)
    all_data.append(df)
    time.sleep(2)  # Add a delay between subreddits to respect rate limits

# Combine all dataframes
teacher_data = pd.concat(all_data, ignore_index=True)

# Convert UTC timestamps to datetime
teacher_data['created_date'] = pd.to_datetime(teacher_data['created_utc'], unit='s')

# Basic data cleanup
teacher_data['text'] = teacher_data['text'].fillna('')  # Replace NaN with empty string
teacher_data['title'] = teacher_data['title'].fillna('')  # Replace NaN with empty string

# Save the data
teacher_data.to_csv('teacher_reddit_data.csv', index=False)
print(f"Collected {len(teacher_data)} posts from {len(subreddits)} subreddits")

In [ ]:
# Basic analysis of the collected data

# Posts per subreddit
subreddit_counts = teacher_data['subreddit'].value_counts()
print("\nPosts per subreddit:")
print(subreddit_counts)

# Posts over time (monthly)
teacher_data['year_month'] = teacher_data['created_date'].dt.to_period('M')
monthly_posts = teacher_data.groupby('year_month').size()

# Plotting the number of posts over time
plt.figure(figsize=(15, 6))
monthly_posts.plot(kind='line')
plt.title('Number of Teacher-Related Posts Over Time')
plt.xlabel('Date')
plt.ylabel('Number of Posts')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# Most discussed topics (based on post titles)
from collections import Counter
import re

def get_common_words(text_series, min_length=4, top_n=20):
    # Combine all text
    text = ' '.join(text_series.astype(str))
    # Convert to lowercase and split into words
    words = re.findall(r'\w+', text.lower())
    # Filter words by length and remove common stop words
    stop_words = {'this', 'that', 'have', 'just', 'from', 'like', 'with', 'what', 'about', 'would'}
    words = [w for w in words if len(w) >= min_length and w not in stop_words]
    # Get word frequencies
    return Counter(words).most_common(top_n)

# Analyze common words in titles
common_title_words = get_common_words(teacher_data['title'])
print("\nMost common words in post titles:")
for word, count in common_title_words:
    print(f"{word}: {count}")